# L2d: Building and Testing a Defensive Fibonacci Program

The Fibonacci recurrence is simple. A dependable callable interface also requires explicit decisions about indexing, representation, invalid input, and integer overflow.

> **Learning Objectives**
> 1. Convert a recurrence into an iterative algorithm.
> 2. Define indexing and return representation explicitly.
> 3. Test base cases, ordinary cases, invalid inputs, and numerical limits.
> 4. Explain why defensive checks belong at the public interface.

---

## Setup, Data, and Prerequisites

Run the local setup cell first. It activates the single pinned course environment, loads every package used by this meeting, and includes any meeting source code.

In [ ]:
include(joinpath(@__DIR__, "Include.jl"))

## Mathematical definition

We use $F_0=0$, $F_1=1$, and $F_n=F_{n-1}+F_{n-2}$ for $n\ge2$. The function returns the complete vector `[F₀, F₁, …, Fₙ]`, so Julia array position `n+1` stores $F_n$.

## Algorithm before syntax

1. Validate that `n` is an integer in the supported range.
2. Allocate `n+1` integer positions.
3. Store the base cases that are needed.
4. Fill each later position from the preceding two positions.
5. Return the populated vector.

In [ ]:
sequence = fibonacci_sequence(10)
(sequence = sequence, F10 = sequence[10 + 1])

## Base cases are ordinary supported inputs

Early returns keep the implementation from writing `F₁` when the requested output ends at `F₀`.

In [ ]:
base_cases = (n0 = fibonacci_sequence(0), n1 = fibonacci_sequence(1))

## Representation changes the access pattern

The vector is compact and ordered. A dictionary can preserve mathematical zero-based keys, but adds hashing and storage overhead that this sequential algorithm does not require.

In [ ]:
dictionary_view = Dict((position - 1) => value for (position, value) in enumerate(sequence))
(vector_F7 = sequence[8], dictionary_F7 = dictionary_view[7])

## Reject unsupported states

Negative and non-integer indices do not belong to this interface. `Int64` can store $F_{92}$ but not $F_{93}$, so the public contract stops at 92 instead of silently overflowing.

In [ ]:
largest_supported = last(fibonacci_sequence(92))
overflow_message = try
    fibonacci_sequence(93)
    "no error"
catch error
    sprint(showerror, error)
end
(F92 = largest_supported, F93 = overflow_message)

## Test the complete contract

In [ ]:

@testset "defensive Fibonacci program" begin
    @test base_cases.n0 == [0]
    @test base_cases.n1 == [0, 1]
    @test sequence == [0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55]
    @test dictionary_view[10] == 55
    @test largest_supported == 7_540_113_804_746_346_429
    @test_throws ArgumentError fibonacci_sequence(-1)
    @test_throws ArgumentError fibonacci_sequence(2.5)
    @test_throws ArgumentError fibonacci_sequence(93)
end

## Summary

> **Key Takeaways**
> 1. Indexing and output representation are part of the interface.
> 2. Base cases and invalid cases deserve tests, not informal assumptions.
> 3. Numerical type limits can define a legitimate API boundary.
> 4. Recursive Fibonacci is deferred to the Week 3 recursion comparison.

---